# Evaluation Metrics Master Guide (Career Edition)

This is your long-term reference for model evaluation. It explains what each metric means, when to use it, why it matters, and common mistakes.

---

## 1) First Principles: What Evaluation Really Means

Evaluation is not about getting one high number. It is about answering:

1. Is the model solving the right business or product problem?
2. Is it reliable on unseen data?
3. Is it safe and fair enough for deployment?
4. Is improvement statistically meaningful or just noise?

Always define these before training:

- Problem type: classification, regression, ranking, forecasting, clustering, detection, generation.
- Success metric: the metric that decides go or no-go.
- Guardrail metrics: metrics that must not degrade (latency, fairness, false negative rate, cost).
- Slice metrics: performance by segment (country, gender, device, new users, rare classes).

---

## 2) Core Classification Metrics

### 2.1 Confusion Matrix (Binary)

- Definition: a table of TP, TN, FP, and FN that shows how predictions compare to actual labels.
- TP: predicted positive, actually positive.
- TN: predicted negative, actually negative.
- FP: predicted positive, actually negative.
- FN: predicted negative, actually positive.

Most classification metrics come from these 4 numbers.

### 2.2 Accuracy

- Definition: the overall fraction of predictions that are correct.

```text
Accuracy = (TP + TN) / (TP + TN + FP + FN)
```

- Why it matters: simple and intuitive overall correctness.
- Use when: classes are balanced and FP/FN have similar cost.
- Pitfall: misleading on imbalanced datasets.

Example: in fraud detection with 99% non-fraud, always predicting non-fraud gives 99% accuracy and is useless.

### 2.3 Precision (Positive Predictive Value)

- Definition: among predicted positives, the fraction that is actually positive.

```text
Precision = TP / (TP + FP)
```

- Why it matters: among predicted positives, how many are truly positive.
- Use when: false positives are expensive (spam flagging important email, wrongful blocking).

### 2.4 Recall (Sensitivity, TPR)

- Definition: among actual positives, the fraction correctly identified by the model.

```text
Recall = TP / (TP + FN)
```

- Why it matters: among actual positives, how many did you catch.
- Use when: false negatives are expensive (disease detection, fraud misses).

### 2.5 Specificity (TNR)

- Definition: among actual negatives, the fraction correctly identified as negative.

```text
Specificity = TN / (TN + FP)
```

- Why it matters: ability to correctly reject negatives.
- Often paired with recall in medical and risk systems.

### 2.6 F1 Score and F-beta

- Definition: combined precision-recall score; F1 balances both equally, F-beta lets you prioritize one.

$$
F1 = 2\cdot\frac{\text{Precision}\cdot\text{Recall}}{\text{Precision} + \text{Recall}}
$$

$$
F_\beta = (1+\beta^2)\cdot\frac{PR}{\beta^2P + R}
$$

- Why it matters: balances precision and recall.
- Use when: classes are imbalanced and you need one score.
- $\beta > 1$: emphasize recall; $\beta < 1$: emphasize precision.

### 2.7 Balanced Accuracy

- Definition: average of sensitivity and specificity, designed for imbalanced classes.

```text
Balanced Accuracy = (TPR + TNR) / 2
```

- Why it matters: robust to class imbalance.

### 2.8 MCC (Matthews Correlation Coefficient)

- Definition: a balanced correlation-style score of binary classification quality using all confusion matrix terms.

$$
MCC = \frac{TP\cdot TN - FP\cdot FN}{\sqrt{(TP+FP)(TP+FN)(TN+FP)(TN+FN)}}
$$

- Range: -1 to +1.
- Why it matters: strong single metric for imbalanced binary classification.

### 2.9 ROC-AUC

- Definition: threshold-independent measure of how well the model ranks positives above negatives.
- ROC curve plots TPR vs FPR across thresholds.
- AUC is threshold-independent ranking quality.

```text
FPR = FP / (FP + TN)
```

- Why it matters: compares models independent of one threshold.
- Pitfall: can look optimistic under heavy imbalance.

### 2.10 PR-AUC (Average Precision)

- Definition: area under precision-recall curve, summarizing precision-recall trade-off across thresholds.
- Precision vs Recall across thresholds.
- Better than ROC-AUC for rare positive class problems.

### 2.11 Log Loss (Cross-Entropy)

- Definition: average penalty on predicted probabilities, with high penalty for confident wrong predictions.

```text
LogLoss = -(1/N) * SUM[ y_i*log(p_i) + (1 - y_i)*log(1 - p_i) ]
```

- Why it matters: evaluates probability quality, not just class labels.
- Strong penalty for confident wrong predictions.

### 2.12 Brier Score (Calibration Error in Probabilities)

- Definition: mean squared difference between predicted probabilities and actual outcomes.

```text
Brier = (1/N) * SUM[ (p_i - y_i)^2 ]
```

- Why it matters: measures probabilistic accuracy and calibration.

### 2.13 Top-k Accuracy

- Definition: prediction is counted correct if true class appears within top k predicted classes.
- Correct if true label appears in top k predictions.
- Common in multi-class vision and recommendation candidates.

---

## 3) Multi-class and Multi-label Details

For precision/recall/F1 in multi-class:

- Micro average: global TP/FP/FN across all classes. Favors frequent classes.
- Macro average: average metric per class equally. Good for class balance awareness.
- Weighted average: per-class metric weighted by class support.

For multi-label tasks:

- Hamming loss: fraction of wrong labels.
- Subset accuracy: exact label set match (very strict).
- Jaccard index: overlap of predicted and true label sets.

---

## 4) Regression Metrics

### 4.1 MAE (Mean Absolute Error)

- Definition: average absolute distance between actual and predicted values.

$$
MAE = \frac{1}{N}\sum|y - \hat{y}|
$$

- Why it matters: robust, interpretable in target units.
- Less sensitive to outliers than MSE.

### 4.2 MSE (Mean Squared Error)

- Definition: average squared prediction error, emphasizing larger mistakes.

$$
MSE = \frac{1}{N}\sum(y - \hat{y})^2
$$

- Why it matters: heavily penalizes large errors.
- Use when large mistakes are very costly.

### 4.3 RMSE (Root Mean Squared Error)

- Definition: square root of MSE, giving error in original target units.

$$
RMSE = \sqrt{MSE}
$$

- Why it matters: same unit as target and still outlier-sensitive.

### 4.4 RMSLE

- Definition: RMSE computed in log space, focusing more on relative differences.

$$
RMSLE = \sqrt{\frac{1}{N}\sum\left(\log(1+\hat{y}) - \log(1+y)\right)^2}
$$

- Why it matters: focuses on relative error and handles large scales better.
- Use for skewed positive targets (sales, counts).

### 4.5 MAPE / sMAPE / WAPE

- Definition: percentage-based error measures that express error relative to actual values.

$$
MAPE = \frac{100}{N}\sum\left|\frac{y-\hat{y}}{y}\right|
$$

- MAPE fails when $y=0$.
- sMAPE reduces instability.
- WAPE is often better for business reporting.

### 4.6 R-squared and Adjusted R-squared

- Definition: R-squared shows explained variance; adjusted R-squared corrects for unnecessary predictors.

$$
R^2 = 1 - \frac{\sum(y-\hat{y})^2}{\sum(y-\bar{y})^2}
$$

- Why it matters: proportion of variance explained.
- Pitfall: high $R^2$ does not imply causal correctness or good generalization.

Adjusted $R^2$ penalizes adding unnecessary features.

### 4.7 Median Absolute Error

- Definition: median of absolute errors, robust to extreme outliers.
- Very robust to outliers.
- Good for heavy-tailed error distributions.

### 4.8 Quantile Loss (Pinball Loss)

- Definition: asymmetric loss used to evaluate quantile predictions such as P50 or P90.
- Used for predicting quantiles, not just mean.
- Essential for uncertainty-aware forecasting.

---

## 5) Forecasting Metrics (Time Series)

### 5.1 MAE, RMSE, MAPE

- Definition: standard point-forecast error metrics measured in absolute, squared-root, and percentage terms.
Still common, but evaluate on proper time splits only.

### 5.2 MASE (Mean Absolute Scaled Error)

- Definition: MAE scaled by naive forecast error to compare across time series.
- Scales error against naive baseline forecast.
- Great for comparing across different series.

### 5.3 sMAPE and WAPE

- Definition: percentage-style forecasting errors that are often more stable than raw MAPE.
- Frequently used in business forecasting dashboards.

### 5.4 Backtesting Rules That Matter

- Never random split time-series data.
- Use rolling or expanding window validation.
- Compare against naive seasonal baseline.

---

## 6) Ranking and Recommendation Metrics

### 6.1 Precision@k and Recall@k

- Definition: top-k ranking metrics measuring relevance quality among shown items and coverage of relevant items.
- Measures quality of top k recommendations.

### 6.2 MAP (Mean Average Precision)

- Definition: average precision over ranked positions, then averaged across users or queries.
- Averaged precision over relevant positions.

### 6.3 NDCG@k (Normalized Discounted Cumulative Gain)

- Definition: ranking metric that rewards placing highly relevant items near the top.
- Rewards ranking relevant items higher in the list.
- Handles graded relevance.

### 6.4 MRR (Mean Reciprocal Rank)

- Definition: average inverse rank of the first relevant result.
- Reciprocal of rank of first relevant result.
- Useful for search and QA retrieval.

### 6.5 HitRate@k

- Definition: fraction of cases where at least one relevant item appears in top k.
- Whether at least one relevant item appears in top k.

Also track product quality metrics:

- Coverage: how much catalog gets recommended.
- Diversity: how varied results are.
- Novelty/Serendipity: not only popular obvious items.

---

## 7) Clustering and Unsupervised Metrics

### Internal Metrics (no ground truth labels)

- Silhouette score: separation vs compactness, range [-1, 1].
- Davies-Bouldin index: lower is better.
- Calinski-Harabasz index: higher is better.
- Inertia (k-means): lower is better but always decreases with k.

### External Metrics (ground truth labels available)

- ARI (Adjusted Rand Index).
- NMI (Normalized Mutual Information).
- Homogeneity, Completeness, V-measure.

---

## 8) Computer Vision Metrics

### 8.1 Detection: IoU, AP, mAP

- Definition: IoU measures overlap quality; AP summarizes precision-recall for one class; mAP averages AP values.
- IoU (Intersection over Union): box overlap quality.
- AP: area under precision-recall curve for a class.
- mAP: mean AP across classes and IoU thresholds.

### 8.2 Segmentation: IoU and Dice

- Definition: overlap metrics that quantify how well predicted masks match true masks.

```text
Dice = 2 * |A_intersect_B| / (|A| + |B|)
```

- Dice often preferred in medical segmentation.

### 8.3 Image Generation

- Definition: FID and IS estimate realism, quality, and diversity of generated images.
- FID: distribution similarity between real and generated images.
- IS (Inception Score): quality and diversity proxy.

---

## 9) NLP and Generative AI Metrics

### NLP Task Metrics

- Classification: accuracy, F1, macro-F1.
- NER: entity-level precision/recall/F1.
- Translation: BLEU, chrF.
- Summarization: ROUGE.
- QA: Exact Match, token-level F1.

### Language Modeling / LLM Metrics

- Perplexity: token prediction uncertainty.
- Pass@k (code generation): probability of at least one correct output in k tries.
- Groundedness/Faithfulness: response aligns with source facts.
- Hallucination rate: unsupported statements frequency.
- Safety metrics: toxicity, bias, harmful output rate.

Rule: automated metrics are not enough for GenAI. Human evaluation and task success rate are essential.

---

## 10) Calibration and Thresholding

Good classifiers need both ranking quality and calibrated probabilities.

- Calibration tools: reliability diagram, Expected Calibration Error (ECE), Brier score.
- Threshold selection based on cost matrix, not default 0.5.

Example decision policy:

- If false negatives cost 10x false positives, choose lower threshold to increase recall.

---

## 11) Fairness and Responsible Evaluation

Measure by slices, not only global average.

Common fairness views:

- Demographic parity.
- Equal opportunity (equal TPR across groups).
- Equalized odds (equal TPR and FPR).
- Calibration within groups.

Real systems require trade-offs and policy context.

---

## 12) Statistical Confidence and Significance

Never trust a single run.

- Use cross-validation where appropriate.
- Report mean +- std (or confidence intervals).
- Bootstrap confidence intervals for robust uncertainty.
- Significance tests:
- Classification paired outputs: McNemar test.
- AUC comparison: DeLong test.
- Paired metric samples: paired t-test or Wilcoxon.

For online systems:

- A/B test with predefined primary metric, power, and stopping criteria.

---

## 13) Choosing the Right Metric Quickly

Use this practical mapping:

- Imbalanced classification: PR-AUC, F1/F-beta, MCC, recall at fixed precision.
- Medical or safety-critical classification: recall, specificity, sensitivity at operating threshold.
- Probability quality needed: log loss, Brier, calibration error.
- Regression with outliers: MAE or Median AE.
- Regression where large errors hurt badly: RMSE.
- Forecasting across many series: MASE, WAPE, sMAPE.
- Ranking/recommendation: NDCG@k, MAP, MRR, Recall@k.
- Clustering without labels: silhouette + domain validation.
- Object detection: mAP at relevant IoU thresholds.
- LLM systems: task success + factuality + safety + latency/cost.

---

## 14) Common Career-Level Mistakes to Avoid

1. Optimizing a metric that does not match business impact.
2. Reporting only one metric and hiding trade-offs.
3. Ignoring class imbalance and data drift.
4. Using random split for time-series data.
5. Comparing models without statistical confidence.
6. Not evaluating by user segments (slice blindness).
7. No baseline comparisons (naive, heuristic, previous model).
8. Ignoring calibration when probabilities drive decisions.
9. Offline wins but online losses due to distribution shift.

---

## 15) What Interviewers and Real Jobs Expect You To Know

You should be able to explain clearly:

1. Why accuracy fails on imbalanced data.
2. Precision vs recall trade-off with a real example.
3. ROC-AUC vs PR-AUC and when each is better.
4. Why MAE and RMSE choose different models.
5. How to choose threshold from business costs.
6. How to evaluate time-series without leakage.
7. How to report uncertainty (CI, CV, significance).
8. Why fairness and slice metrics matter in production.
9. Why offline metric uplift may not guarantee product uplift.

---

## 16) Final Practical Checklist (Before You Ship a Model)

- Primary metric defined and justified.
- Baseline beaten by meaningful margin.
- Guardrail metrics safe.
- Segment-wise evaluation done.
- Calibration checked if probabilities are used.
- Threshold chosen using cost/risk.
- Uncertainty/statistical significance reported.
- Data leakage and drift checks completed.
- Monitoring plan for production metrics prepared.

If you remember only one thing: a metric is useful only when it represents real-world decision quality.

## Quick Meaning Of Every Metric (Plain English)

Use this as a fast dictionary when you forget what a metric means.

### Classification

- **Accuracy**: how many total predictions are correct.
- **Precision**: of predicted positives, how many are truly positive.
- **Recall (Sensitivity/TPR)**: of actual positives, how many are detected.
- **Specificity (TNR)**: of actual negatives, how many are correctly rejected.
- **F1 Score**: balance of precision and recall.
- **F-beta**: like F1, but can favor recall or precision.
- **Balanced Accuracy**: average of recall and specificity.
- **MCC**: correlation-style quality score for binary prediction.
- **ROC-AUC**: ability to rank positives above negatives over all thresholds.
- **PR-AUC**: precision-recall quality over thresholds; best for imbalanced positives.
- **Log Loss**: penalty for wrong predicted probabilities (especially confident wrong ones).
- **Brier Score**: squared error of predicted probabilities.
- **Top-k Accuracy**: correct if true class appears in top k guesses.

### Multi-class and Multi-label

- **Micro average**: one global metric using all TP/FP/FN together.
- **Macro average**: per-class metric, all classes weighted equally.
- **Weighted average**: per-class metric weighted by class frequency.
- **Hamming Loss**: fraction of labels predicted incorrectly.
- **Subset Accuracy**: exact match of entire label set.
- **Jaccard Index**: overlap between predicted and true label sets.

### Regression and Forecasting

- **MAE**: average absolute error in original units.
- **MSE**: average squared error (punishes large errors heavily).
- **RMSE**: square root of MSE (same unit as target).
- **RMSLE**: log-scale error; focuses more on relative error.
- **MAPE**: average percentage error.
- **sMAPE**: stabilized percentage error.
- **WAPE**: total absolute error divided by total actual value.
- **R-squared**: fraction of variance explained.
- **Adjusted R-squared**: R-squared adjusted for number of features.
- **Median Absolute Error**: median of absolute errors, robust to outliers.
- **Quantile (Pinball) Loss**: quality of quantile prediction (P50/P90 etc.).
- **MASE**: error scaled against naive forecast error.

### Ranking and Recommendation

- **Precision@k**: relevant items among top k shown.
- **Recall@k**: fraction of all relevant items found in top k.
- **MAP**: mean average precision across users/queries.
- **NDCG@k**: ranking quality with extra credit for higher positions.
- **MRR**: average inverse rank of first relevant result.
- **HitRate@k**: whether at least one relevant item appears in top k.
- **Coverage**: how much of catalog gets recommended.
- **Diversity**: how varied recommended items are.
- **Novelty/Serendipity**: how useful and non-obvious recommendations are.

### Clustering

- **Silhouette Score**: how well each point fits its own cluster vs others.
- **Davies-Bouldin**: cluster overlap/similarity (lower is better).
- **Calinski-Harabasz**: between-cluster separation relative to within-cluster compactness.
- **Inertia**: total squared distance to assigned centroid.
- **ARI**: cluster-label agreement corrected for chance.
- **NMI**: normalized shared information between cluster labels and true labels.
- **Homogeneity**: each cluster has mostly one class.
- **Completeness**: each class is mostly in one cluster.
- **V-measure**: balance of homogeneity and completeness.

### Vision, NLP, and GenAI

- **IoU**: overlap between predicted and true box/region.
- **AP**: area under precision-recall for one class.
- **mAP**: average AP across classes (and IoU levels).
- **Dice**: overlap score for segmentation masks.
- **FID**: generated-vs-real image distribution distance.
- **IS**: quality-diversity proxy for generated images.
- **BLEU**: n-gram precision overlap with reference.
- **chrF**: character-level F-score for translation.
- **ROUGE**: overlap recall metrics for summarization.
- **Exact Match**: exact answer match rate.
- **Token F1**: token overlap quality for QA.
- **Perplexity**: language model uncertainty.
- **Pass@k**: chance at least one of k outputs is correct.
- **Groundedness/Faithfulness**: response consistency with source evidence.
- **Hallucination Rate**: unsupported statement frequency.
- **Safety Metrics**: toxic/biased/harmful output rates.

### Calibration, Fairness, and Statistics

- **Calibration**: predicted probabilities match real frequencies.
- **Reliability Diagram**: visual calibration check by confidence bins.
- **ECE**: average confidence-vs-accuracy gap.
- **Demographic Parity**: equal positive prediction rates across groups.
- **Equal Opportunity**: equal recall across groups.
- **Equalized Odds**: equal recall and false positive rates across groups.
- **Calibration Within Groups**: calibration holds per group.
- **Cross-Validation**: repeated splits for robust performance estimate.
- **Confidence Interval**: likely range of true metric.
- **Bootstrap CI**: confidence interval via resampling.
- **McNemar Test**: paired classification significance test.
- **DeLong Test**: significance test for ROC-AUC difference.
- **Paired t-test**: mean paired difference significance test.
- **Wilcoxon Signed-Rank**: non-parametric paired significance test.
- **A/B Test**: online controlled comparison between versions.